In [ ]:
import logging
from src.config import load_config
from src.data.loader import load_datasets
from src.models.codegen_loader_small_model import CodeGenModelWrapper
from src.models.finetuned_LoRa import train_codegen_lora
from src.generators.program_generator import run_program_gen
from src.generators.doc_generator import run_doc_gen
from src.generators.sql_generator import run_text_to_sql
from src.generators.commit_generator import run_commit_gen
from src.indexing.embeddings import generate_code_embeddings
from src.indexing.semantic_index import FAISSIndexManager
from src.indexing.ast_index import ASTIndexManager
from src.rag.rag_retriever import TopKRetriever
from src.rag.rag_pipeline import RAGPipeline
from src.evaluation.comparator import compare_architectures
from src.evaluation.visualization import save_plots

In [ ]:
logging.basicConfig(level=logging.INFO, format="%(asctime)s - [%(levelname)s] - %(message)s")

In [ ]:
def main():
    print("=" * 70)
    print("STARTING END-TO-END QWEN CODER PIPELINE")
    print("=" * 70)

    # 1. Load Configuration
    config = load_config()

    # 2. Load Datasets (Online benchmarks & nested local JSONL files in data/datasets/)
    datasets = UnifiedDatasetLoader.load_datasets(config)
    logging.info(f"Loaded {len(datasets.get('local', {}))} local dataset files/folders.")

    # 3. Model Setup (Salesforce/codegen-350M-multi with LoRA)
    model = CodeGenMultiModelWrapper(
        model_name=config["model"]["name_or_path"],
        device=config["project"]["device"],
        use_lora=True,
        lora_kwargs=config.get("lora", {})
    )

    # 4. Fine-Tune Pass via LoRA
    training_texts = ["def binary_search(arr, t): return -1"]
    train_codegen_lora(model, training_texts, config["outputs"]["checkpoints_dir"], epochs=1)

    # 5. Task Output Generation using untouched generator modules
    p_out = program_generator.synthesize(model, problem="Return the maximum value in a list of integers.", language="python")
    d_out = doc_generator.generate_doc(model, code="def calc_area(r):\n    return 3.14 * r ** 2")
    s_out = sql_generator.generate_sql(model, schema="CREATE TABLE users(id INT, status TEXT);", question="Find all active users")
    c_out = commit_generator.generate_commit_message(model, diff="--- a/file.py\n+++ b/file.py\n@@ -1 +1 @@\n-print(1)\n+print(2)")

    print(f"[Generated Program]: {p_out[0] if p_out else ''}")
    print(f"[Generated Docstring]: {d_out}")
    print(f"[Generated SQL]: {s_out}")
    print(f"[Generated Commit Message]: {c_out}")

    # 6. Indexing (FAISS & AST Tree-Sitter)
    code_corpus = [
        "def quicksort(arr): return arr",
        "SELECT * FROM users WHERE active = 1;",
        "def add(a, b): return a + b"
    ]
    embeddings = generate_code_lm_embeddings(model, code_corpus)

    faiss_mgr = FAISSIndexManager()
    faiss_mgr.build_index(embeddings, code_corpus)

    ast_mgr = TreeSitterASTIndexer(model)
    ast_mgr.build_ast_index(code_corpus)

    # 7. RAG Pipeline Execution
    retriever = TopKRetriever(model, faiss_mgr)
    rag_pipe = RAGPipeline(model)

    query = "Write a Python function for sorting"
    retrieved_ctx = retriever.retrieve(query, top_k=config["indexing"]["top_k"])
    rag_result = rag_pipe.execute(query, retrieved_ctx)

    print("\n--- FINAL RAG GENERATED OUTPUT ---")
    print(rag_result)
    print("----------------------------------\n")

    # 8. Evaluation & Chart Visualization
    comparison = ArchitectureComparator.compare_architectures()
    ResultsVisualizer.save_metrics(comparison, f"{config['outputs']['metrics_dir']}/results.json")
    ResultsVisualizer.generate_chart(comparison, config['outputs']['plots_dir'])

    print("=" * 70)
    print("PIPELINE EXECUTION COMPLETE")
    print("=" * 70)

if __name__ == "__main__":
    main()

In [ ]:
if __name__ == "__main__":
    main()